# DuckDB Read-only Extraction for Requested Metrics

This notebook extracts the requested variables from DuckDB in **read-only mode** using your measurement definitions.
It uses the same database path as the app (`config.DB_PATH`) so results match the Data Studio page.

- `EDC individual items`: checklist binary indicators returned as separate columns (`CC1`...`ACC2`)
- `SIZE`: natural logarithm of total assets
- `BOARDSIZE`: total number of directors on the board
- `AGE`: firm age (years since establishment proxy)
- `ROA`: net income / total assets
- `LOA`: current liabilities / total assets (alternative proxy to reduce missing values)
- `ISO`: dummy = 1 if ISO 14001 is present, else 0
- `WOMAN`: dummy = 1 if at least one female director exists, else 0

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

try:
    from config import DB_PATH as APP_DB_PATH
    DB_PATH = Path(APP_DB_PATH)
    db_source = 'config.DB_PATH'
except Exception:
    DB_PATH = Path('/media/nvme0n1/dev/annual_report/db.db')
    db_source = 'fallback literal path'

assert DB_PATH.exists(), f'Database file not found: {DB_PATH}'

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute('PRAGMA threads=4;')
print('Connected to', DB_PATH)
print('DB source:', db_source)
print(con.execute("SELECT current_database(), current_schema()").fetchall())

Connected to /media/nvme0n1/dev/annual_report/db.db
DB source: config.DB_PATH
[('db', 'main')]


In [2]:
# Check required tables first so the extraction cell can fail fast with a clear message.
required_tables = [
    'financial_models',
    'financial_statements',
    'governance_results',
    'proper_vn_results',
    'inference_results',
    'company_history_summary',
]

tables_df = con.execute(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_name
    """
).df()
available_tables = set(tables_df['table_name'].tolist())
missing = [t for t in required_tables if t not in available_tables]

print('Available tables:', len(available_tables))
display(tables_df.head(50))

if missing:
    print('Missing required tables:', missing)
else:
    print('All required tables are present.')

Available tables: 26


,table_name
0,annual_reports
1,bctc_audit_results
2,bctc_document_embeddings
3,bctc_reports
4,companies
5,company_history_events
6,company_history_summary
7,conversion_jobs
8,document_embeddings
9,financial_models


All required tables are present.


In [3]:
sql = """
WITH code_pool AS (
    SELECT DISTINCT
        item_code,
        lower(COALESCE(item_en_name, '')) AS en_name,
        lower(COALESCE(item_vn_name, '')) AS vn_name
    FROM financial_models
),
code_scored AS (
    SELECT
        item_code,
        CASE
            WHEN en_name = 'total assets' OR vn_name = 'tổng tài sản' THEN 'total_assets'
            WHEN en_name LIKE '%total assets%' OR vn_name LIKE '%tổng tài sản%' THEN 'total_assets'
            WHEN en_name LIKE '%profit after tax%' OR en_name LIKE '%net income%' OR vn_name LIKE '%lợi nhuận sau thuế%' THEN 'net_income'
            WHEN en_name = 'current liabilities' OR vn_name = 'nợ ngắn hạn' THEN 'current_liabilities'
            WHEN en_name LIKE '%current liabilities%' OR vn_name LIKE '%nợ ngắn hạn%' THEN 'current_liabilities'
            ELSE NULL
        END AS metric,
        CASE
            WHEN en_name = 'total assets' OR vn_name = 'tổng tài sản' THEN 300
            WHEN en_name LIKE '%total assets%' OR vn_name LIKE '%tổng tài sản%' THEN 250
            WHEN en_name LIKE '%profit after tax%' OR vn_name LIKE '%lợi nhuận sau thuế%' THEN 300
            WHEN en_name LIKE '%net income%' THEN 250
            WHEN en_name = 'current liabilities' OR vn_name = 'nợ ngắn hạn' THEN 300
            WHEN en_name LIKE '%current liabilities%' OR vn_name LIKE '%nợ ngắn hạn%' THEN 250
            ELSE 0
        END AS priority
    FROM code_pool
),
code_map AS (
    SELECT
        metric,
        arg_max(item_code, priority) AS item_code
    FROM code_scored
    WHERE metric IS NOT NULL
    GROUP BY metric
),
fs_base AS (
    SELECT
        fs.code AS ticker,
        TRY_CAST(SUBSTR(fs.fiscal_date, 1, 4) AS INTEGER) AS year,
        fs.item_code,
        fs.numeric_value,
        fs.fiscal_date,
        fs.modified_date
    FROM financial_statements fs
    JOIN code_map cm
      ON fs.item_code = cm.item_code
    WHERE fs.numeric_value IS NOT NULL
) ,
fs_latest AS (
    SELECT
        ticker,
        year,
        item_code,
        numeric_value
    FROM fs_base
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ticker, year, item_code
        ORDER BY fiscal_date DESC, modified_date DESC
    ) = 1
) ,
firm_metrics AS (
    SELECT
        l.ticker,
        l.year,
        MAX(CASE WHEN l.item_code = (SELECT item_code FROM code_map WHERE metric = 'total_assets') THEN l.numeric_value END) AS total_assets,
        MAX(CASE WHEN l.item_code = (SELECT item_code FROM code_map WHERE metric = 'net_income') THEN l.numeric_value END) AS net_income,
        MAX(CASE WHEN l.item_code = (SELECT item_code FROM code_map WHERE metric = 'current_liabilities') THEN l.numeric_value END) AS current_liabilities
    FROM fs_latest l
    GROUP BY l.ticker, l.year
) ,
edc_items AS (
    SELECT
        ticker,
        year,
        model,
        MAX(CASE WHEN category_code = 'CC1' THEN CAST(is_valid AS INTEGER) END) AS cc1,
        MAX(CASE WHEN category_code = 'CC2' THEN CAST(is_valid AS INTEGER) END) AS cc2,
        MAX(CASE WHEN category_code = 'GHG1' THEN CAST(is_valid AS INTEGER) END) AS ghg1,
        MAX(CASE WHEN category_code = 'GHG2' THEN CAST(is_valid AS INTEGER) END) AS ghg2,
        MAX(CASE WHEN category_code = 'GHG3' THEN CAST(is_valid AS INTEGER) END) AS ghg3,
        MAX(CASE WHEN category_code = 'GHG4' THEN CAST(is_valid AS INTEGER) END) AS ghg4,
        MAX(CASE WHEN category_code = 'GHG5' THEN CAST(is_valid AS INTEGER) END) AS ghg5,
        MAX(CASE WHEN category_code = 'GHG6' THEN CAST(is_valid AS INTEGER) END) AS ghg6,
        MAX(CASE WHEN category_code = 'GHG7' THEN CAST(is_valid AS INTEGER) END) AS ghg7,
        MAX(CASE WHEN category_code = 'EC1' THEN CAST(is_valid AS INTEGER) END) AS ec1,
        MAX(CASE WHEN category_code = 'EC2' THEN CAST(is_valid AS INTEGER) END) AS ec2,
        MAX(CASE WHEN category_code = 'EC3' THEN CAST(is_valid AS INTEGER) END) AS ec3,
        MAX(CASE WHEN category_code = 'RC1' THEN CAST(is_valid AS INTEGER) END) AS rc1,
        MAX(CASE WHEN category_code = 'RC2' THEN CAST(is_valid AS INTEGER) END) AS rc2,
        MAX(CASE WHEN category_code = 'RC3' THEN CAST(is_valid AS INTEGER) END) AS rc3,
        MAX(CASE WHEN category_code = 'RC4' THEN CAST(is_valid AS INTEGER) END) AS rc4,
        MAX(CASE WHEN category_code = 'ACC1' THEN CAST(is_valid AS INTEGER) END) AS acc1,
        MAX(CASE WHEN category_code = 'ACC2' THEN CAST(is_valid AS INTEGER) END) AS acc2
    FROM inference_results
    WHERE category_code IN ('ACC1','ACC2','CC1','CC2','EC1','EC2','EC3','GHG1','GHG2','GHG3','GHG4','GHG5','GHG6','GHG7','RC1','RC2','RC3','RC4')
    GROUP BY ticker, year, model
) ,
gov_base AS (
    SELECT DISTINCT ticker, year, model
    FROM governance_results
) ,
gov_dir AS (
    SELECT ticker, year, model, details_json
    FROM governance_results
    WHERE item_code = 'GOV_DIRECTORY'
) ,
proper_items AS (
    SELECT
        ticker,
        year,
        model,
        MAX(CASE WHEN indicator_code = 'S1_VIOLATION' AND is_present THEN 1 ELSE 0 END) AS proper_s1_violation,
        MAX(CASE WHEN indicator_code = 'S1_MINOR_NC' AND is_present THEN 1 ELSE 0 END) AS proper_s1_minor_nc,
        MAX(CASE WHEN indicator_code = 'S1_COMPLIANCE' AND is_present THEN 1 ELSE 0 END) AS proper_s1_compliance,
        MAX(CASE WHEN indicator_code = 'S2_ISO14001' AND is_present THEN 1 ELSE 0 END) AS proper_s2_iso14001,
        MAX(CASE WHEN indicator_code = 'S2_CARBON_DISC' AND is_present THEN 1 ELSE 0 END) AS proper_s2_carbon_disc,
        MAX(CASE WHEN indicator_code = 'S2_REDUCTION' AND is_present THEN 1 ELSE 0 END) AS proper_s2_reduction,
        MAX(CASE WHEN indicator_code = 'S2_EFFICIENCY' AND is_present THEN 1 ELSE 0 END) AS proper_s2_efficiency
    FROM proper_vn_results
    GROUP BY ticker, year, model
) ,
gov_profile AS (
    SELECT
        b.ticker,
        b.year,
        b.model,
        COALESCE(json_array_length(COALESCE(d.details_json, '[]')), 0) AS boardsize,
        CASE
            WHEN ch.first_event_year IS NOT NULL THEN b.year - ch.first_event_year
            ELSE NULL
        END AS age,
        COALESCE(p.proper_s2_iso14001, 0) AS iso,
        COALESCE(p.proper_s1_violation, 0) AS proper_s1_violation,
        COALESCE(p.proper_s1_minor_nc, 0) AS proper_s1_minor_nc,
        COALESCE(p.proper_s1_compliance, 0) AS proper_s1_compliance,
        COALESCE(p.proper_s2_iso14001, 0) AS proper_s2_iso14001,
        COALESCE(p.proper_s2_carbon_disc, 0) AS proper_s2_carbon_disc,
        COALESCE(p.proper_s2_reduction, 0) AS proper_s2_reduction,
        COALESCE(p.proper_s2_efficiency, 0) AS proper_s2_efficiency,
        CASE
            WHEN COALESCE((
                SELECT COUNT(*)
                FROM json_each(COALESCE(d.details_json, '[]')) jd
                WHERE lower(trim(COALESCE(json_extract_string(jd.value, '$.gender'), ''))) IN ('female', 'nữ', 'nu')
            ), 0) > 0 THEN 1
            ELSE 0
        END AS woman
    FROM gov_base b
    LEFT JOIN gov_dir d
      ON d.ticker = b.ticker AND d.year = b.year AND d.model = b.model
    LEFT JOIN proper_items p
      ON p.ticker = b.ticker AND p.year = b.year AND p.model = b.model
    LEFT JOIN company_history_summary ch
      ON ch.ticker = b.ticker
)
SELECT
    g.ticker,
    g.year,
    g.model,
    COALESCE(e.cc1, 0) AS cc1,
    COALESCE(e.cc2, 0) AS cc2,
    COALESCE(e.ghg1, 0) AS ghg1,
    COALESCE(e.ghg2, 0) AS ghg2,
    COALESCE(e.ghg3, 0) AS ghg3,
    COALESCE(e.ghg4, 0) AS ghg4,
    COALESCE(e.ghg5, 0) AS ghg5,
    COALESCE(e.ghg6, 0) AS ghg6,
    COALESCE(e.ghg7, 0) AS ghg7,
    COALESCE(e.ec1, 0) AS ec1,
    COALESCE(e.ec2, 0) AS ec2,
    COALESCE(e.ec3, 0) AS ec3,
    COALESCE(e.rc1, 0) AS rc1,
    COALESCE(e.rc2, 0) AS rc2,
    COALESCE(e.rc3, 0) AS rc3,
    COALESCE(e.rc4, 0) AS rc4,
    COALESCE(e.acc1, 0) AS acc1,
    COALESCE(e.acc2, 0) AS acc2,
    LN(NULLIF(f.total_assets, 0)) AS size,
    g.boardsize,
    g.age,
    f.net_income / NULLIF(f.total_assets, 0) AS roa,
    f.current_liabilities / NULLIF(f.total_assets, 0) AS loa,
    g.iso,
    g.proper_s1_violation,
    g.proper_s1_minor_nc,
    g.proper_s1_compliance,
    g.proper_s2_iso14001,
    g.proper_s2_carbon_disc,
    g.proper_s2_reduction,
    g.proper_s2_efficiency,
    g.woman
FROM gov_profile g
LEFT JOIN firm_metrics f
  ON f.ticker = g.ticker AND f.year = g.year
LEFT JOIN edc_items e
  ON e.ticker = g.ticker AND e.year = g.year AND e.model = g.model
WHERE g.ticker in 
('ACV',
'ASG',
'BLN',
'CCP',
'CCR',
'CCT',
'CDN',
'CLL',
'CQN',
'DDH',
'DL1',
'DNL',
'DVP',
'DXP',
'EMS',
'GIC',
'HAH',
'HMH',
'HNB',
'HTV',
'ILC',
'MVN',
'NCT',
'NWT',
'PCT',
'PDN',
'PHP',
'PNP',
'PSN',
'PSP',
'PTT',
'QNP',
'QSP',
'RAT',
'SAC',
'SAS',
'SCS',
'SFI',
'SGN',
'SGP',
'SGS',
'SHC',
'STG',
'STS',
'TAB',
'TCL',
'TCO',
'TCT',
'TCW',
'TMS',
'TNP',
'TPS',
'TR1',
'TUG',
'VFC',
'VGP',
'VGR',
'VIN',
'VMS',
'VMT',
'VNF',
'VNL',
'VSA',
'VSC',
'VSE',
'VSM',
'VTP',
'VXT',
'WCS',
'WTC')
ORDER BY g.ticker, g.year DESC, g.model
"""

df = con.execute(sql).df()
print('Rows:', len(df))
print('LOA missing:', int(df['loa'].isna().sum()))
display(df.head(100))

Rows: 328
LOA missing: 0


,ticker,year,model,cc1,cc2,ghg1,ghg2,ghg3,ghg4,ghg5,...,loa,iso,proper_s1_violation,proper_s1_minor_nc,proper_s1_compliance,proper_s2_iso14001,proper_s2_carbon_disc,proper_s2_reduction,proper_s2_efficiency,woman
0,ASG,2025,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.223096,1,0,0,0,1,0,0,0,1
1,ASG,2024,gpt-4.1-mini,1,0,0,0,0,0,0,...,0.299956,0,0,0,0,0,0,0,0,1
2,ASG,2023,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.220352,0,0,0,0,0,0,0,0,1
3,ASG,2022,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.193414,0,0,1,0,0,0,0,0,1
4,ASG,2021,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.194006,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,HMH,2019,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.064574,0,0,0,0,0,0,0,0,0
96,HMH,2018,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.051899,0,0,0,0,0,0,0,0,0
97,HMH,2017,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.056626,0,0,0,0,0,0,0,0,0
98,HMH,2016,gpt-4.1-mini,0,0,0,0,0,0,0,...,0.077937,0,0,0,0,0,0,0,0,0


In [4]:
# Optional export
out_path = Path('data/output/requested_items_metrics.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print('Saved to', out_path.resolve())

Saved to /media/nvme0n1/dev/annual_report/data/output/requested_items_metrics.csv
